# Adapter smoke test (throwaway, PRIVATE)
Same 3 tasks (fastapi, rich, requests), one model turn each, two vLLM passes started like the scorer does it:
**A** no adapter, **B** with the adapter (`adapters/main_lora`). No docker, no grading. Prints adapter facts, the LoRA
lines of the vLLM log, and per task: finish reason, token count, whether a well-formed tool call appeared, first 500 chars.
Raw outputs go to `/kaggle/working/raw_A.json` / `raw_B.json`. Keep this notebook private; delete it afterwards.

In [ ]:
import os, sys, json, re, shutil, subprocess, hashlib, struct, time, zipfile, importlib.util
from pathlib import Path
RUN_OVERRIDES = {}  # @@RUN_OVERRIDES@@
REPOS = ["fastapi/fastapi", "Textualize/rich", "psf/requests"]
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working"); TMP = Path("/tmp/smoke"); TMP.mkdir(exist_ok=True, parents=True)

def walk(pred, root=INPUT, depth=7):
    hits = []
    for dp, dn, fn in os.walk(root, followlinks=True):
        d = Path(dp)
        if pred(d, set(fn)):
            hits.append(d); dn[:] = []; continue
        dn[:] = [] if len(d.relative_to(root).parts) >= depth else sorted(dn)
    return hits

In [ ]:
# 1) offline install, same recipe as the training notebook (official wheelhouse first, then ours, both --no-deps)
whs = walk(lambda d, f: any(n.endswith(".whl") for n in f), depth=4)
print("wheelhouses:", [str(w) for w in whs])
def install(wh, force):
    dst = Path("/tmp/wh") / wh.name; dst.mkdir(parents=True, exist_ok=True)
    files = []
    for w in sorted(wh.glob("*.whl")):
        n = w.name.replace("cu128", "+cu128", 1) if (w.name.startswith("torch-") and "cu128" in w.name and "+cu128" not in w.name) else w.name
        if not (dst / n).exists(): shutil.copy2(w, dst / n)
        files.append(str(dst / n))
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", str(dst), "--no-deps"] + (["--force-reinstall"] if force else [])
    lock = wh / "requirements.lock.txt"
    items = [l.strip() for l in lock.read_text().splitlines() if l.strip() and not l.startswith("#")] if lock.is_file() else files
    r = subprocess.run(cmd + items); print(wh.name, "pip rc", r.returncode)
official = [w for w in whs if "developer-agent" in str(w).lower()]
for w in official: install(w, True)
for w in whs:
    if w not in official and 'tpu' not in w.name.lower(): install(w, False)  # skip the TPU wheelhouse: libtpu makes vLLM detect a tpu platform
# our wheelhouse (TPU training stack) drags in a vLLM TPU platform plugin -> "Only one platform plugin can be activated: cuda, tpu". Remove it.
from importlib import metadata as _md
_pk = ["libtpu", "libtpu-nightly", "tpu-info", "torch-xla", "torch_xla", "jax", "jaxlib", "tpu-inference", "vllm-tpu"]
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q"] + _pk)
_bad = {e.dist.name for e in _md.entry_points(group="vllm.platform_plugins") if e.name != "cuda" and e.dist is not None}
print("non-cuda vllm platform plugins:", _bad)
if _bad: subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q"] + sorted(_bad))
print("remaining platform plugins:", [(e.name, e.dist.name) for e in _md.entry_points(group="vllm.platform_plugins")])
r = subprocess.run([sys.executable, "-c", "import vllm, adk_submission, swegemma; print('vllm', vllm.__version__)"], capture_output=True, text=True)
print(r.stdout or r.stderr[-1500:])

In [ ]:
# 2) locate base model, adapter, prompts, tools, tasks
base_dirs = walk(lambda d, f: "config.json" in f and "w4a16" in str(d).lower(), depth=8)
BASE = base_dirs[0]; print("base model:", BASE)
ads = walk(lambda d, f: "adapter_config.json" in f, depth=8)
print("adapters found:", [str(a) for a in ads])
assert ads, "no adapter_config.json under /kaggle/input: attach the LoRA adapter dataset"
AD = next((a for a in ads if "main" in a.name.lower()), ads[0]); print("using adapter dir:", AD)

# adapter facts (no weights printed)
cfgj = json.loads((AD / "adapter_config.json").read_text())
print({k: cfgj.get(k) for k in ("r", "lora_alpha", "target_modules", "base_model_name_or_path", "peft_type", "use_rslora", "modules_to_save")})
st = AD / "adapter_model.safetensors"
h = hashlib.sha256(st.read_bytes()).hexdigest(); print("file", st.name, st.stat().st_size, "bytes sha256", h[:16])
with open(st, "rb") as f:
    n = struct.unpack("<Q", f.read(8))[0]; hdr = json.loads(f.read(n))
hdr.pop("__metadata__", None)
keys = sorted(hdr); print("tensors:", len(keys), "dtypes:", sorted({v["dtype"] for v in hdr.values()}))
print("first keys:", keys[:4])
# do adapter module paths exist in the base model? (wrong prefixes are a classic silent no-op / load reject)
idx = BASE / "model.safetensors.index.json"
if idx.is_file():
    base_keys = set(json.loads(idx.read_text())["weight_map"])
else:
    base_keys = set()
    for sf in BASE.glob("*.safetensors"):
        with open(sf, "rb") as f:
            m = struct.unpack("<Q", f.read(8))[0]; base_keys |= set(json.loads(f.read(m)))
mods = {re.sub(r"\.lora_[AB](\.default)?\.weight$", "", k) for k in keys}
mods = {re.sub(r"^base_model\.model\.", "", m) for m in mods}
hit = sum(1 for m in mods if (m + ".weight") in base_keys)
print(f"adapter modules: {len(mods)}, found in base weight names: {hit}; example module: {sorted(mods)[0]}; example base key: {sorted(base_keys)[0] if base_keys else None}")

# system prompt from the zip we submit (fall back to any prompts/system.md)
SYSTEM = None
for z in walk(lambda d, f: "submission.zip" in f, depth=6):
    with zipfile.ZipFile(z / "submission.zip") as zf:
        if "prompts/system.md" in zf.namelist():
            SYSTEM = zf.read("prompts/system.md").decode(); print("system prompt from", z / "submission.zip"); break
if SYSTEM is None:
    for d in walk(lambda d, f: (d / "prompts" / "system.md").is_file(), depth=6):
        SYSTEM = (d / "prompts" / "system.md").read_text(); print("system prompt from", d); break
if SYSTEM is None:
    SYSTEM = "You are a software engineering agent. Fix the issue in /workspace using the tools, then call submit_patch."; print("WARNING: fallback system prompt")

# tools: the repo's mirror of the 9 harness tools
tp = next(INPUT.rglob("agent/tools.py"))
spec = importlib.util.spec_from_file_location("tools_mod", tp); tm = importlib.util.module_from_spec(spec); spec.loader.exec_module(tm)
TOOLS = [{"type": "function", "function": {"name": t["name"], "description": t["description"], "parameters": t["parameters"]}} for t in tm.OFFICIAL_HOST_TOOLS.values()]
print("tools:", [t["function"]["name"] for t in TOOLS])

# tasks: first task of each wanted repo (contents are not printed)
tf = next(INPUT.rglob("tasks.jsonl")); chosen = {}
for line in tf.read_text().splitlines():
    t = json.loads(line)
    if t["repo"] in REPOS and t["repo"] not in chosen: chosen[t["repo"]] = t
tasks = [{"id": t["instance_id"], "repo": r, "prompt": f"Repository {r} is checked out at /workspace.\n\nIssue:\n{t['problem_statement']}\n\nFix it using the tools, then call submit_patch."} for r, t in chosen.items()]
print("tasks:", [(t["id"], t["repo"]) for t in tasks]); assert len(tasks) == 3
(TMP / "tasks.json").write_text(json.dumps(tasks)); (TMP / "tools.json").write_text(json.dumps(TOOLS)); (TMP / "system.md").write_text(SYSTEM)

# staged submissions: A has no adapters/, B has adapters/main_lora
for lab in "AB":
    d = TMP / f"sub_{lab}"; shutil.rmtree(d, ignore_errors=True); d.mkdir()
    (d / "agent.yaml").write_text("name: swe_agent\nmodel: gemma-4-31b-it-qat-w4a16-ct\n" + ("adapter: main_lora\n" if lab == "B" else ""))
shutil.copytree(AD, TMP / "sub_B" / "adapters" / "main_lora", ignore=shutil.ignore_patterns("optimizer*", "*.pt", "*.bin"))
RANK = int(cfgj.get("r", 128))

In [ ]:
Path("/tmp/smoke_pass.py").write_text(r"""
import argparse, json, os, time, urllib.request, traceback
ap = argparse.ArgumentParser()
for k in ("label", "sub", "model", "out"): ap.add_argument("--" + k)
ap.add_argument("--adapter", default=""); ap.add_argument("--rank", type=int, default=128); a = ap.parse_args()
os.environ.setdefault("VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS", "1")
import adk_submission, litellm
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
tasks = json.load(open("/tmp/smoke/tasks.json")); tools = json.load(open("/tmp/smoke/tools.json")); system = open("/tmp/smoke/system.md").read()
names = {t["function"]["name"] for t in tools}
res = {"label": a.label, "tasks": [], "error": None}
server = None
try:
    man = adk_submission.discover_adapters(a.sub, adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
    ads = set(getattr(man, "adapters", None) or {}); print("ADAPTERS DISCOVERED", sorted(ads), flush=True)
    cfg = adk_submission.VllmConfig(model=a.model, host="127.0.0.1", port=8000, tool_call_parser="gemma4", reasoning_parser="gemma4",
        max_model_len=32768, dtype="bfloat16", gpu_memory_utilization=0.90, enable_auto_tool_choice=True, enable_lora=bool(ads),
        max_loras=max(1, len(ads)), max_lora_rank=a.rank if ads else 128, scheduler_reserve_full_isl=False,
        tensor_parallel_size=4, startup_timeout=2400, extra_args=[])
    server = adk_submission.VllmServer(cfg, adapter_manifest=man)
    t0 = time.time(); server.start(); print("VLLM READY after %.0fs" % (time.time() - t0), flush=True)
    ids = [m["id"] for m in json.load(urllib.request.urlopen("http://127.0.0.1:8000/v1/models"))["data"]]; print("SERVED MODEL IDS", ids, flush=True)
    target = a.adapter if a.adapter else a.model
    if a.adapter and a.adapter not in ids: raise RuntimeError("adapter %r not in served model ids %s" % (a.adapter, ids))
    for t in tasks:
        body = {"model": target, "messages": [{"role": "system", "content": system}, {"role": "user", "content": t["prompt"]}], "tools": tools,
                "temperature": 0.2, "top_p": 0.95, "max_tokens": 8192, "thinking_token_budget": 4096, "chat_template_kwargs": {"enable_thinking": True}}
        r = {"id": t["id"], "repo": t["repo"]}
        try:
            req = urllib.request.Request("http://127.0.0.1:8000/v1/chat/completions", json.dumps(body).encode(), {"Content-Type": "application/json"})
            t1 = time.time(); resp = json.load(urllib.request.urlopen(req, timeout=1800)); r["sec"] = round(time.time() - t1, 1)
            ch = resp["choices"][0]; m = ch["message"]; tcs = m.get("tool_calls") or []
            ok = []
            for tc in tcs:
                try: json.loads(tc["function"]["arguments"] or "{}"); ok.append(tc["function"]["name"] in names)
                except Exception: ok.append(False)
            content = m.get("content") or ""
            r.update(finish_reason=ch.get("finish_reason"), completion_tokens=resp.get("usage", {}).get("completion_tokens"),
                     tool_calls=[tc["function"]["name"] for tc in tcs], well_formed_tool_call=bool(ok) and all(ok),
                     raw_markup_in_text=any(s in content for s in ("<|tool_call", "tool_call>", "<tool_call")),
                     reasoning_chars=len(m.get("reasoning_content") or m.get("reasoning") or ""), first500=content[:500], raw=resp)
        except Exception as e:
            r["error"] = repr(e)[:500]
        res["tasks"].append(r)
        print("TASK", {k: v for k, v in r.items() if k not in ("raw", "first500")}, flush=True)
except Exception:
    res["error"] = traceback.format_exc()[-3000:]; print(res["error"], flush=True)
finally:
    json.dump(res, open(a.out, "w"), indent=1)
    try:
        if server: server.stop()
    except Exception as e: print("stop warning", e)
""")

def run_pass(label, adapter):
    log = WORK / f"vllm_{label}.log"
    cmd = [sys.executable, "/tmp/smoke_pass.py", "--label", label, "--sub", str(TMP / f"sub_{label}"), "--model", str(BASE),
           "--out", str(WORK / f"raw_{label}.json"), "--rank", str(RANK)] + (["--adapter", adapter] if adapter else [])
    with open(log, "w") as lf:
        p = subprocess.run(cmd, stdout=lf, stderr=subprocess.STDOUT, text=True)
    txt = log.read_text(errors="replace")
    print(f"\n===== pass {label} exit {p.returncode}; log {len(txt)} chars =====")
    print("-- LoRA / adapter lines from the vLLM log --")
    for l in txt.splitlines():
        if re.search(r"lora|adapter", l, re.I): print(l[:300])
    print("-- worker lines --")
    for l in txt.splitlines():
        if l.startswith(("ADAPTERS DISCOVERED", "VLLM READY", "SERVED MODEL IDS", "TASK ")) or "Traceback" in l: print(l[:600])
    return json.loads((WORK / f"raw_{label}.json").read_text()) if (WORK / f"raw_{label}.json").exists() else None

In [ ]:
RA = run_pass("A", "")
time.sleep(30)  # let the GPUs free up before the second server
RB = run_pass("B", "main_lora")

In [ ]:
for R in (RA, RB):
    print("\n##### pass", R and R["label"], "error:", R and R["error"])
    for t in (R or {}).get("tasks", []):
        print({k: t.get(k) for k in ("id", "repo", "finish_reason", "completion_tokens", "tool_calls", "well_formed_tool_call", "raw_markup_in_text", "reasoning_chars", "sec", "error")})
        print("   first500:", repr(t.get("first500", ""))[:560])
print("\nRead: B errors/rejects -> packaging; B runs to token cap or no valid tool calls while A is fine -> trained weights; both fine -> Kaggle side.")